# Disaster Risk Data: US Per-State CSV Generator

Generates per-state CSV files with FEMA NRI county-level hazard risk scores
for all 50 US states and DC. Data source: **[FEMA National Risk Index (NRI)](https://www.fema.gov/about/openfema/data-sets/national-risk-index-data)**:
  18 natural-hazard risk scores for every US county (December 2025, v3).

See `CA-MX_disaster_risk_analysis.ipynb` for Canadian and Mexican equivalents (mocked).

## Output

Per-state CSV files written to `../plugins/emfn-action-pack-plugin/assets/data/`:

| Region | Files | Source |
|--------|-------|--------|
| US states + DC | `AL.csv` … `WY.csv` (51 files) | NRI real data |

Each file has columns: `county_fips`, `state`, `county`, `{HAZARD}_risk_score` × 18.


In [ ]:
# Install all project dependencies declared in pyproject.toml into the venv.
# Skip if you have already run `uv sync` from the repo root.
!uv sync

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from shared_setup import check_dependencies
check_dependencies()

In [ ]:
import tempfile, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

## US — FEMA National Risk Index

The NRI ZIP (~50 MB) is downloaded once and cached at `cache/NRI_Table_Counties.csv`.
Re-running uses the cached file. Delete the cache file to force a fresh download.

In [ ]:
# Paths
NRI_ZIP_URL   = "https://www.fema.gov/about/reports-and-data/openfema/nri/v120/NRI_Table_Counties.zip"
NRI_CSV_CACHE = Path("cache/NRI_Table_Counties.csv")
OUTPUT_DIR    = Path("../plugins/emfn-action-pack-plugin/assets/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NRI hazard codes (December 2025, v3) — order matches AL.csv sample
HAZARD_CODES = [
    "AVLN", "CFLD", "CWAV", "DRGT", "ERQK", "HAIL",
    "HWAV", "HRCN", "ISTM", "LNDS", "LTNG", "IFLD",
    "SWND", "TRND", "TSUN", "VLCN", "WFIR", "WNTW",
]

# USPS 2-letter state codes (50 states + DC)
US_STATES = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

print(f"US states: {len(US_STATES)}")
print(f"Hazard types: {len(HAZARD_CODES)}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")


In [ ]:
def load_nri_data() -> pd.DataFrame:
    """Return the NRI county table, downloading from FEMA and caching locally on first run."""
    if NRI_CSV_CACHE.exists():
        print(f"Loading cached NRI data from {NRI_CSV_CACHE} …")
        return pd.read_csv(NRI_CSV_CACHE, dtype={"STCOFIPS": str})

    print(f"Downloading NRI data from FEMA …")
    NRI_CSV_CACHE.parent.mkdir(parents=True, exist_ok=True)

    temp_zip_path = None
    with requests.get(NRI_ZIP_URL, stream=True, timeout=300) as resp:
        resp.raise_for_status()

        total = int(resp.headers.get("content-length", 0))
        try:
            with tempfile.NamedTemporaryFile(
                suffix=".zip",
                delete=False,
                dir=NRI_CSV_CACHE.parent,
            ) as tmp_zip:
                temp_zip_path = Path(tmp_zip.name)
                with tqdm(total=total, unit="B", unit_scale=True, desc="Downloading") as bar:
                    for chunk in resp.iter_content(chunk_size=65536):
                        if not chunk:
                            continue
                        tmp_zip.write(chunk)
                        bar.update(len(chunk))

            with zipfile.ZipFile(temp_zip_path) as zf:
                csv_name = next(
                    n for n in zf.namelist()
                    if n.endswith(".csv") and "NRI_Table_Counties" in n
                )
                print(f"Extracting {csv_name} …")
                with zf.open(csv_name) as f:
                    df = pd.read_csv(f, dtype={"STCOFIPS": str})
        finally:
            if temp_zip_path and temp_zip_path.exists():
                temp_zip_path.unlink()

    df.to_csv(NRI_CSV_CACHE, index=False)
    print(f"Cached to {NRI_CSV_CACHE}")
    return df


nri_raw = load_nri_data()
print(f"\nLoaded {len(nri_raw):,} counties · {nri_raw.shape[1]} columns")
print("State column candidates:", [c for c in nri_raw.columns if "STATE" in c.upper()])

In [ ]:
def nri_rows_to_output(df: pd.DataFrame) -> pd.DataFrame:
    """Convert a slice of the raw NRI table to the output CSV schema.

    Output columns: county_fips, state, county, {HAZARD}_risk_score × 18.
    Handles the RFLD→IFLD rename between NRI v2 and v3.
    """
    out = pd.DataFrame({
        "county_fips": df["STCOFIPS"],
        "state":       df["STATE"],
        "county":      df["COUNTY"],
    })
    for haz in HAZARD_CODES:
        src = f"{haz}_RISKS"
        if haz == "IFLD" and src not in df.columns and "RFLD_RISKS" in df.columns:
            src = "RFLD_RISKS"   # NRI v2 used RFLD; v3 uses IFLD
        out[f"{haz}_risk_score"] = df[src].values if src in df.columns else pd.NA
    return out.reset_index(drop=True)


# Identify the state-abbreviation column (varies by NRI version)
abbr_col = next(
    (c for c in ("STATEABBRV", "STUSPS", "STATE_ABBRV") if c in nri_raw.columns),
    None,
)
if abbr_col is None:
    nri_raw = nri_raw.copy()
    nri_raw["_ABBR"] = nri_raw["STATE"].map(US_STATES)
    abbr_col = "_ABBR"

written = 0
valid_abbrs = set(US_STATES.values())
for abbr, group in tqdm(nri_raw.groupby(abbr_col), desc="Writing US state CSVs"):
    if pd.isna(abbr) or abbr not in valid_abbrs:
        continue
    out_df = nri_rows_to_output(group)
    (OUTPUT_DIR / f"{abbr}.csv").write_text(out_df.to_csv(index=False))
    written += 1

print(f"\nWrote {written} US state CSVs to {OUTPUT_DIR}/")

## Summary

In [ ]:
all_csvs = sorted(OUTPUT_DIR.glob("*.csv"))
us = [f for f in all_csvs if f.stem in set(US_STATES.values())]

print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"  US state CSVs: {len(us):3d}  ({', '.join(f.stem for f in us[:6])} …)")


### Percentage Cut-off Analysis

Explore CSV output for # of counties (grouped by state) where a cut-off of >=X.00 means we show no risk data.

This analysis helps determine optimal threshold values by showing how many counties would have **zero displayable risks** at various cut-offs.

In [ ]:
# Configure the display threshold (percentile score)
DISPLAY_THRESHOLD = 85.0  # Counties with all hazards < this value show no risks

# Analyze counties with zero displayable risks
zero_risk_counties = []
state_total_counties = {}

for csv_path in us:
    abbr = csv_path.stem
    df = pd.read_csv(csv_path, dtype={"county_fips": str})
    
    # Track total counties per state
    state_total_counties[abbr] = len(df)
    
    # Get all risk score columns
    risk_cols = [c for c in df.columns if c.endswith("_risk_score")]
    
    # For each county, count how many risks are >= threshold
    for _, row in df.iterrows():
        scores = pd.to_numeric(row[risk_cols], errors="coerce")
        above_threshold = (scores >= DISPLAY_THRESHOLD).sum()
        
        if above_threshold == 0:
            max_score = scores.max() if not scores.isna().all() else 0.0
            # Find which hazard has the max score
            max_risk_code = None
            if not scores.isna().all():
                max_idx = scores.idxmax()
                # Extract hazard code from column name (e.g., "AVLN_risk_score" -> "AVLN")
                max_risk_code = max_idx.replace("_risk_score", "")
            
            zero_risk_counties.append({
                "state": abbr,
                "county": row["county"],
                "county_fips": row["county_fips"],
                "max_score": max_score,
                "max_risk_code": max_risk_code,
            })

# Create summary DataFrame
zero_df = pd.DataFrame(zero_risk_counties)

# Group by state and count
if not zero_df.empty:
    state_summary = zero_df.groupby("state").size().reset_index(name="zero_risk_counties")
    
    # Add total counties and percentage columns
    state_summary["total_counties"] = state_summary["state"].map(state_total_counties)
    state_summary["pct_of_counties"] = (state_summary["zero_risk_counties"] / state_summary["total_counties"] * 100).round(1)
    
    state_summary = state_summary.sort_values("pct_of_counties", ascending=False)
    total_zero = len(zero_df)
    total_counties = sum(state_total_counties.values())
    pct_zero = (total_zero / total_counties) * 100
    
    print(f"Threshold: >={DISPLAY_THRESHOLD:.1f} percentile")
    print(f"Total counties with ZERO displayable risks: {total_zero:,} ({pct_zero:.1f}%)")
    print(f"\nBreakdown by state (ordered by counties with zero risks, descending):")
    print(state_summary.to_string(index=False))
else:
    print(f"Threshold: >={DISPLAY_THRESHOLD:.1f} percentile")
    print("✅ All counties have at least one risk above the threshold")

Data analysis: if we cut the RiskRenderer user-facing output to just 3 visible risks, how many counties would have:
* show 3 100% risks
* hide other 100% risks 
* cut off X risks above #sym:riskThreshold

In [ ]:
# RiskRenderer top-3 cap analysis at current #sym:riskThreshold (DISPLAY_THRESHOLD)
# Analyzes what happens when we display only the top 3 risks (sorted descending by score)

threshold = DISPLAY_THRESHOLD  # reuses threshold already defined above

total_counties = 0
counties_top3_all_100 = 0
counties_cutting_off_100s = 0
counties_cutting_off_99_to_threshold = 0

rows = []

for csv_path in us:
    abbr = csv_path.stem
    df = pd.read_csv(csv_path, dtype={"county_fips": str})
    risk_cols = [c for c in df.columns if c.endswith("_risk_score")]

    for _, row in df.iterrows():
        scores = pd.to_numeric(row[risk_cols], errors="coerce").dropna()
        total_counties += 1

        # Sort risks descending by score
        sorted_scores = scores.sort_values(ascending=False)
        
        # Top 3 risks shown to user
        top3 = sorted_scores.head(3)
        # Risks cut off (4th onwards)
        cut_off_risks = sorted_scores.iloc[3:]
        
        # Extract risk names from column names (e.g., "HRCN_risk_score" -> "HRCN")
        top3_names = [col.replace("_risk_score", "") for col in top3.index]
        cut_off_names = [col.replace("_risk_score", "") for col in cut_off_risks.index]
        
        # Stats: top-3 all 100% (check for >= 99.5 since scores round to 100% when displayed)
        top3_all_100 = (len(top3) == 3) and (top3.min() >= 99.5)
        if top3_all_100:
            counties_top3_all_100 += 1
        
        # Stats: cutting off 100% risks (>= 99.5 rounds to 100% when displayed)
        cut_off_100s = cut_off_risks[cut_off_risks >= 99.5]
        if len(cut_off_100s) > 0:
            counties_cutting_off_100s += 1
        
        # Stats: cutting off 99% down to threshold
        cut_off_99_to_threshold = cut_off_risks[(cut_off_risks >= threshold) & (cut_off_risks < 100)]
        if len(cut_off_99_to_threshold) > 0:
            counties_cutting_off_99_to_threshold += 1
        
        # Track high-value cutoffs (95%+)
        cut_off_95_plus = cut_off_risks[cut_off_risks >= 95.0]
        
        rows.append({
            "state": abbr,
            "county_fips": row["county_fips"],
            "county": row["county"],
            "top3_all_100": top3_all_100,
            "num_cutoff_100s": len(cut_off_100s),
            "num_cutoff_99_to_threshold": len(cut_off_99_to_threshold),
            "num_cutoff_95_plus": len(cut_off_95_plus),
            "top3_risks": ", ".join([f"{name} ({score:.0f}%)" for name, score in zip(top3_names, top3.values)]),
            "cutoff_100s": ", ".join([f"{col.replace('_risk_score', '')}" for col in cut_off_100s.index]) if len(cut_off_100s) > 0 else "",
            "cutoff_95_plus": ", ".join([f"{col.replace('_risk_score', '')} ({score:.0f}%)" for col, score in zip(cut_off_95_plus.index, cut_off_95_plus.values)]) if len(cut_off_95_plus) > 0 else "",
            "max_cutoff_score": cut_off_risks.max() if len(cut_off_risks) > 0 else 0.0,
        })

top3_df = pd.DataFrame(rows)

# ── Summary Statistics ──────────────────────────────────────────────────────────
print(f"Threshold (#sym:riskThreshold): >= {threshold:.1f}")
print(f"Total counties analyzed: {total_counties:,}\n")

print("📊 Top-3 Display Cap Impact:")
print(f"  1️⃣  Counties where top-3 risks are all 100%: {counties_top3_all_100:,} ({counties_top3_all_100/total_counties*100:.1f}%)")
print(f"  2️⃣  Counties cutting off other 100% risks: {counties_cutting_off_100s:,} ({counties_cutting_off_100s/total_counties*100:.1f}%)")
print(f"  3️⃣  Counties cutting off 99%–{threshold:.0f}% risks: {counties_cutting_off_99_to_threshold:,} ({counties_cutting_off_99_to_threshold/total_counties*100:.1f}%)")

# ── Sample: Counties Cutting Off High Risks (95%+) ─────────────────────────────
print(f"\n📍 Sample: Counties cutting off 95%+ risks (top 30 by max cutoff score):")
high_cutoff_sample = top3_df[top3_df["num_cutoff_95_plus"] > 0].sort_values("max_cutoff_score", ascending=False).head(30)

if not high_cutoff_sample.empty:
    display_cols = high_cutoff_sample[["state", "county", "top3_risks", "cutoff_95_plus"]]
    print(display_cols.to_string(index=False, max_colwidth=80))
else:
    print("  ✅ No counties are cutting off 95%+ risks")

# ── Detailed: Counties Cutting Off 100% Risks ──────────────────────────────────
print(f"\n🚨 Counties cutting off 100% risks (top 30 by # of 100% cutoffs):")
cutoff_100s_sample = top3_df[top3_df["num_cutoff_100s"] > 0].sort_values("num_cutoff_100s", ascending=False).head(30)

if not cutoff_100s_sample.empty:
    display_cols = cutoff_100s_sample[["state", "county", "num_cutoff_100s", "top3_risks", "cutoff_100s"]]
    print(display_cols.to_string(index=False, max_colwidth=80))
else:
    print("  ✅ No counties are cutting off 100% risks")

#### Skippable

In [ ]:
# Display zero-risk counties for a specific state
# Change STATE_ABBR to any state code (e.g., "CA", "TX", "NY")
STATE_ABBR = "RI"

if not zero_df.empty:
    state_counties = zero_df[zero_df["state"] == STATE_ABBR]
    
    if not state_counties.empty:
        print(f"Counties in {STATE_ABBR} with zero displayable risks (threshold >={DISPLAY_THRESHOLD:.1f}):")
        print(f"Total: {len(state_counties)} counties\n")
        display_cols = state_counties[["county", "county_fips", "max_score", "max_risk_code"]].sort_values("max_score", ascending=True)
        print(display_cols.to_string(index=False))
    else:
        print(f"✅ No counties in {STATE_ABBR} have zero displayable risks at threshold >={DISPLAY_THRESHOLD:.1f}")
else:
    print("No counties with zero displayable risks across all states")

### Integration tests

Validates that every generated CSV satisfies the schema contract expected by the JS client (`emfn-action-pack-plugin.js`).

| Test | What it checks |
|------|---------------|
| T1 | All 51 state CSVs exist |
| T2 | Column names match the JS-expected set exactly (`county_fips`, `state`, `county`, 18 × `{HAZ}_risk_score`) |
| T3 | `county_fips` is always a 5-digit zero-padded string (required for the JS FIPS lookup) |
| T4 | No duplicate `county_fips` within a single state file |
| T5 | All hazard score columns are numeric or blank — no stray strings that would make `parseFloat` return `NaN` |
| T6 | Present scores are within the valid `[0, 100]` percentile range |

In [ ]:
import gc
import re as _re

# Hazard codes the JS client maps via NRI_HAZARD_LABELS — order must match emfn-action-pack-plugin.js
JS_HAZARD_CODES = [
    "AVLN", "CFLD", "CWAV", "DRGT", "ERQK", "HAIL",
    "HWAV", "HRCN", "ISTM", "LNDS", "LTNG", "IFLD",
    "SWND", "TRND", "TSUN", "VLCN", "WFIR", "WNTW",
]
EXPECTED_COLS  = ["county_fips", "state", "county"] + [f"{h}_risk_score" for h in JS_HAZARD_CODES]
FIPS_RE        = _re.compile(r"^\d{5}$")
EXPECTED_ABBRS = set(US_STATES.values())

failures = []

# T1 – every expected state CSV exists
missing_files = EXPECTED_ABBRS - {f.stem for f in all_csvs}
if missing_files:
    failures.append(f"T1 FAIL – missing CSV(s): {sorted(missing_files)}")
else:
    print(f"T1 PASS  all {len(EXPECTED_ABBRS)} state CSVs present")

# T2–T6 – per-file schema and data integrity
for csv_path in us:
    abbr = csv_path.stem
    try:
        df = pd.read_csv(csv_path, dtype={"county_fips": str})
    except Exception as exc:
        failures.append(f"T2 FAIL [{abbr}] unreadable: {exc}")
        continue

    # T2 – column names match JS-expected schema
    missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing_cols:
        failures.append(f"T2 FAIL [{abbr}] missing columns: {missing_cols}")

    # T3 – county_fips is always a 5-digit zero-padded string
    present  = df["county_fips"].dropna()
    bad_fips = present[~present.str.match(FIPS_RE)]
    if not bad_fips.empty:
        failures.append(f"T3 FAIL [{abbr}] malformed county_fips: {bad_fips.tolist()[:5]}")

    # T4 – no duplicate FIPS within a state file
    dupes = df.loc[df["county_fips"].duplicated(), "county_fips"]
    if not dupes.empty:
        failures.append(f"T4 FAIL [{abbr}] duplicate county_fips: {dupes.tolist()}")

    # T5 + T6 – score columns: numeric-or-blank, within [0, 100]
    score_cols = [c for c in EXPECTED_COLS if c.endswith("_risk_score") and c in df.columns]
    for col in score_cols:
        vals    = df[col].dropna()                       # non-null values, intact index
        numeric = pd.to_numeric(vals, errors="coerce")   # same index; NaN where non-numeric
        bad_str = vals[numeric.isna()]                   # T5: entries that couldn't be parsed
        if not bad_str.empty:
            failures.append(f"T5 FAIL [{abbr}][{col}] non-numeric: {bad_str.tolist()[:3]}")
        oor = numeric[(numeric < 0) | (numeric > 100)]   # T6: out-of-range scores
        if not oor.empty:
            failures.append(f"T6 FAIL [{abbr}][{col}] out-of-range scores: {oor.tolist()[:3]}")

    del df
    gc.collect()  # release each DataFrame's cyclic refs synchronously

# ── Report ──────────────────────────────────────────────────────────────────────────────────
print()
if failures:
    for msg in failures:
        print(msg)
    raise AssertionError(f"{len(failures)} integration test failure(s) — see above")
else:
    print(f"✅  All {len(us)} state CSVs passed T1–T6")
